In [1]:
# ==========================================
#  UNIFIED MODEL TRAINING & METRICS
# ==========================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import joblib 
import os

# Import all 5 Algorithms
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

def train_and_evaluate_all(csv_path, models_file='all_team_models.pkl'):
    # Check if models are already trained and saved
    if os.path.exists(models_file):
        print(f"Saved models found! Loading '{models_file}' instantly...")
        return joblib.load(models_file)
    
    print("No saved models found. Loading data and training all 5 algorithms...")
    try:
        # Load and standardize data
        df = pd.read_csv(csv_path, encoding='latin-1')
        df = df[['v1', 'v2']] 
        df.columns = ['label', 'text']
        
        X = df['text']
        y = df['label']
        
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # Define the 5 Models (Using pipelines so each model manages its own TF-IDF)
        models = {
            "Algo 1: Naive Bayes": make_pipeline(TfidfVectorizer(stop_words='english'), MultinomialNB()),
            "Algo 2: Logistic Regression": make_pipeline(TfidfVectorizer(), LogisticRegression()),
            "Algo 3: SVM": make_pipeline(TfidfVectorizer(), SVC()),
            "Algo 4: Random Forest": make_pipeline(TfidfVectorizer(), RandomForestClassifier(random_state=42)),
            "Algo 5: Deep Learning (MLP)": make_pipeline(TfidfVectorizer(stop_words='english'), MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42))
        }

        trained_models = {}
        
        # Plotting Grid for Confusion Matrices (2 rows, 3 columns)
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        fig.suptitle("Team Model Performance - Confusion Matrices", fontsize=16, weight='bold')

        print("\n=== FINAL PERFORMANCE REPORT ===")
        
        # Train, Evaluate, and Plot each model
        for idx, (name, pipeline) in enumerate(models.items()):
            print(f"Training {name}...")
            pipeline.fit(X_train, y_train)
            trained_models[name] = pipeline
            
            # Predict and calculate metrics
            y_pred = pipeline.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred, labels=['ham', 'spam'])
            
            print(f"{name} Accuracy: {acc * 100:.2f}%")
            
            # Draw heatmap on the grid
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                        xticklabels=['Pred Ham', 'Pred Spam'], 
                        yticklabels=['Actual Ham', 'Actual Spam'])
            axes[idx].set_title(f"{name}\nAcc: {acc * 100:.2f}%")

        # Hide the empty 6th subplot
        axes[5].set_visible(False)
        plt.tight_layout()
        plt.show() 

        # Save all models together
        joblib.dump(trained_models, models_file)
        print(f"\nAll models saved successfully to '{models_file}'!")
        
        return trained_models

    except FileNotFoundError:
        print(f"Error: Could not find '{csv_path}'.")
        return None

# Execute the master function
team_models = train_and_evaluate_all('spam.csv')

Saved models found! Loading 'all_team_models.pkl' instantly...


In [2]:
# ==========================================
# THE MASTER GUI (ALL MODELS)
# ==========================================
import tkinter as tk
from tkinter import messagebox

def process_master_text():
    if not team_models:
        messagebox.showerror("Error", "Models not loaded. Run Cell 1 first.")
        return

    input_text = text_input.get("1.0", tk.END).strip()
    
    if not input_text:
        messagebox.showwarning("Warning", "Please enter some text to process.")
        return

    # Loop through the UI output boxes and the trained models
    for name, ui_box in output_boxes.items():
        model = team_models[name]
        prediction = model.predict([input_text])[0].upper()
        
        # Update the specific algorithm's text box
        ui_box.config(state='normal')
        ui_box.delete(0, tk.END)
        
        # Add visual flair based on result
        if prediction == "SPAM":
            ui_box.insert(0, "SPAM ❌")
            ui_box.config(fg="red")
        else:
            ui_box.insert(0, "NOT SPAM ✅")
            ui_box.config(fg="green")
            
        ui_box.config(state='readonly')

# Setup the main master window
if team_models:
    root = tk.Tk()
    root.title("Spam Detection - Team Master Project")
    root.geometry("600x450")
    root.configure(bg="#f4f4f4")

    # Top Frame for Input
    input_frame = tk.Frame(root, bg="#f4f4f4")
    input_frame.pack(pady=20, padx=20, fill=tk.X)

    tk.Label(input_frame, text="Input:", font=("Arial", 12, "bold"), bg="#f4f4f4").pack(side=tk.LEFT, anchor=tk.N)
    text_input = tk.Text(input_frame, height=5, width=45, bg="#d9e1f2", font=("Arial", 11))
    text_input.pack(side=tk.LEFT, padx=10)

    process_btn = tk.Button(input_frame, text="Process", bg="#9bc2e6", font=("Arial", 11, "bold"), height=3, width=10, command=process_master_text)
    process_btn.pack(side=tk.LEFT, padx=10)

    # Bottom Frame for Outputs
    output_frame = tk.Frame(root, bg="#f4f4f4")
    output_frame.pack(pady=10, padx=20, fill=tk.BOTH, expand=True)

    output_boxes = {}

    # Helper function to dynamically create the 5 rows
    def create_algo_row(parent, label_text):
        row = tk.Frame(parent, bg="#f4f4f4")
        row.pack(fill=tk.X, pady=8)
        tk.Label(row, text=label_text, width=25, anchor="w", font=("Arial", 11, "bold"), bg="#f4f4f4").pack(side=tk.LEFT)
        entry = tk.Entry(row, bg="#fff2cc", state='readonly', font=("Arial", 11, "bold"))
        entry.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=4)
        return entry

    # Create the 5 UI boxes linked to the model dictionary keys
    output_boxes["Algo 1: Naive Bayes"] = create_algo_row(output_frame, "Algorithm 1 (NB):")
    output_boxes["Algo 2: Logistic Regression"] = create_algo_row(output_frame, "Algorithm 2 (LR):")
    output_boxes["Algo 3: SVM"] = create_algo_row(output_frame, "Algorithm 3 (SVM):")
    output_boxes["Algo 4: Random Forest"] = create_algo_row(output_frame, "Algorithm 4 (RF):")
    output_boxes["Algo 5: Deep Learning (MLP)"] = create_algo_row(output_frame, "Algorithm 5 (Deep Learning):")

    root.mainloop()